## Cat & Dog Image Classification Project (Using Transfer Learning with Pretrained Models)

To speed up training and improve performance, I will use two popular pretrained models:
1. **MobileNetV3Large**: Extremely fast and designed for mobile/edge deployment.
2. **EfficientNetB0**: Highly accurate with a scaling method that balances depth, width, and resolution.

## Importing requirements

In [1]:
import tensorflow
import numpy as np
import pandas as pd

In [2]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 2. Set path to your Drive folder (adjust folder structure if needed)
train_dir = '/content/drive/MyDrive/CNN/train'
test_dir = '/content/drive/MyDrive/CNN/test'

print("Train path exists:", os.path.exists(train_dir))

Train path exists: True


## 1. MobileNetV3Large

In [4]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Large

In [5]:
# 1. Load the MobileNetV3Large base model pre-trained on ImageNet
# We exclude the top classification layer and specify our input shape
base_mobilenet = MobileNetV3Large(input_shape=(128, 128, 3),
                                  include_top=False,
                                  weights='imagenet')

/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:519: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


In [6]:
# Freeze the base model to retain learned features
base_mobilenet.trainable = False

In [7]:
# 2. Build the Model Architecture
mobilenet_model = models.Sequential([
    base_mobilenet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

In [8]:
# 3. Compile the model
mobilenet_model.compile(optimizer='adam',
                        loss='binary_crossentropy',
                        metrics=['accuracy'])

In [9]:
mobilenet_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Large (Functional)   │ (None, 4, 4, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       123,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,119,489 (11.90 MB)

 Trainable params: 123,137 (481.00 KB)

 Non-trainable params: 2,996,352 (11.43 MB)

In [10]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Train MobileNetV3Large
print("Training MobileNetV3Large...")

mobilenet_callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
    ModelCheckpoint('best_mobilenet.keras', monitor='val_accuracy', save_best_only=True)
]

Training MobileNetV3Large...


In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Initialize ImageDataGenerators with rescaling
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

# 2. Flow images from directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

Found 20016 images belonging to 2 classes.
Found 5000 images belonging to 2 classes.


In [12]:
# 3. Fit the model with limited steps to train much faster
mobilenet_history = mobilenet_model.fit(
    train_generator,
    steps_per_epoch=100,      # Only train on 100 batches (3200 images) per epoch
    epochs=15,
    validation_data=test_generator,
    validation_steps=20,      # Only validate on 20 batches (640 images) per epoch
    callbacks=mobilenet_callbacks
)

Epoch 1/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 507s 5s/step - accuracy: 0.5172 - loss: 0.6962 - val_accuracy: 0.5984 - val_loss: 0.6750 - learning_rate: 0.0010
Epoch 2/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 259s 3s/step - accuracy: 0.5666 - loss: 0.6826 - val_accuracy: 0.6281 - val_loss: 0.6647 - learning_rate: 0.0010
Epoch 3/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 235s 2s/step - accuracy: 0.5710 - loss: 0.6742 - val_accuracy: 0.5781 - val_loss: 0.6746 - learning_rate: 0.0010
Epoch 4/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 197s 2s/step - accuracy: 0.5797 - loss: 0.6746 - val_accuracy: 0.5828 - val_loss: 0.6703 - learning_rate: 0.0010
Epoch 5/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 183s 2s/step - accuracy: 0.5925 - loss: 0.6698 - val_accuracy: 0.6250 - val_loss: 0.6520 - learning_rate: 5.0000e-04
Epoch 6/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 159s 2s/step - accuracy: 0.5925 - loss: 0.6643 - val_accuracy: 0.6000 - val_loss: 0.6599 - learning_rate: 5.0000e-04
Epoch 7/15
 24/100 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5425 - 

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


100/100 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - accuracy: 0.5841 - loss: 0.6727 - val_accuracy: 0.5906 - val_loss: 0.6749 - learning_rate: 5.0000e-04
Epoch 8/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 190s 2s/step - accuracy: 0.5966 - loss: 0.6658 - val_accuracy: 0.6125 - val_loss: 0.6576 - learning_rate: 2.5000e-04
Epoch 9/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 132s 1s/step - accuracy: 0.6066 - loss: 0.6616 - val_accuracy: 0.6313 - val_loss: 0.6494 - learning_rate: 2.5000e-04
Epoch 10/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 107s 1s/step - accuracy: 0.6181 - loss: 0.6580 - val_accuracy: 0.6078 - val_loss: 0.6514 - learning_rate: 2.5000e-04
Epoch 11/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 85s 855ms/step - accuracy: 0.6062 - loss: 0.6616 - val_accuracy: 0.6281 - val_loss: 0.6454 - learning_rate: 2.5000e-04
Epoch 12/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 79s 802ms/step - accuracy: 0.5969 - loss: 0.6631 - val_accuracy: 0.6438 - val_loss: 0.6384 - learning_rate: 2.5000e-04
Epoch 13/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 77s 779ms/step - accu

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


100/100 ━━━━━━━━━━━━━━━━━━━━ 39s 393ms/step - accuracy: 0.6202 - loss: 0.6625 - val_accuracy: 0.6109 - val_loss: 0.6536 - learning_rate: 2.5000e-04
Epoch 15/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 62s 628ms/step - accuracy: 0.6256 - loss: 0.6534 - val_accuracy: 0.6422 - val_loss: 0.6417 - learning_rate: 1.2500e-04


### 1.1 Load and Use MobileNetV3Large Pretrained Model (As Is)
Below, we load the model with `include_top=True` to keep its original classification layer trained on ImageNet. We also include a helper function to load, preprocess, and predict any image path.

In [14]:
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# 1. Load MobileNetV3Large fully pre-trained on ImageNet (as is)
pretrained_mobilenet = MobileNetV3Large(weights='imagenet', include_top=True)
print("Model loaded successfully with original classifier layers!")

/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:519: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


22661472/22661472 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model loaded successfully with original classifier layers!


In [15]:
def predict_image(img_path):
    # 2. Load and resize the image to 224x224 (required by default MobileNetV3)
    img = image.load_img(img_path, target_size=(224, 224))

    # 3. Convert image to array and add batch dimension
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)

    # 4. Preprocess input according to MobileNetV3 requirements.txt
    x = preprocess_input(x)

    # 5. Run inference
    preds = pretrained_mobilenet.predict(x)

    # 6. Decode and print the top 3 predictions
    print('Predicted classes:', decode_predictions(preds, top=3)[0])

In [17]:
predict_image('/content/drive/MyDrive/CNN/test/cats/cat.10.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Predicted classes: [('n02123045', 'tabby', np.float32(0.52243984)), ('n02123159', 'tiger_cat', np.float32(0.045229007)), ('n02124075', 'Egyptian_cat', np.float32(0.043819632))]


In [21]:
print("Evaluating the raw, pre-trained ImageNet MobileNetV3 model on your test dataset...")

# ImageNet indices for cats (278 to 287) and dogs (151 to 268)
cat_indices = set(range(278, 288))
dog_indices = set(range(151, 269))

correct_predictions = 0
total_predictions = 0

# Evaluate on 10 batches (320 images) to keep it fast
num_batches_to_eval = 15

for i in range(num_batches_to_eval):
    # Retrieve a batch from test generator (which flows from your drive directory)
    try:
        imgs, labels = next(test_generator)
    except:
        break

    # Raw pre-trained model expects inputs in 224x224 shape and specific preprocessing
    import tensorflow as tf
    resized_imgs = tf.image.resize(imgs * 255.0, (224, 224))  # Scale back to 0-255, resize
    preprocessed_imgs = preprocess_input(resized_imgs.numpy())

    # Make predictions
    preds = pretrained_mobilenet.predict(preprocessed_imgs, verbose=0)

    for j in range(len(labels)):
        # Find the index with the highest probability
        predicted_class_idx = np.argmax(preds[j])

        # Map prediction index to binary class (cat=0, dog=1 based on your directory structure)
        if predicted_class_idx in cat_indices:
            binary_pred = 0
        elif predicted_class_idx in dog_indices:
            binary_pred = 1
        else:
            # If it's outside dogs/cats, assign it to the class with higher sum probability
            cat_prob = sum(preds[j][idx] for idx in cat_indices if idx < len(preds[j]))
            dog_prob = sum(preds[j][idx] for idx in dog_indices if idx < len(preds[j]))
            binary_pred = 0 if cat_prob >= dog_prob else 1

        if binary_pred == int(labels[j]):
            correct_predictions += 1
        total_predictions += 1

Evaluating the raw, pre-trained ImageNet MobileNetV3 model on your test dataset...


In [22]:
accuracy = (correct_predictions / total_predictions) * 100 if total_predictions > 0 else 0
print(f"\nEvaluated Images: {total_predictions}")
print(f"Pre-trained Model Accuracy on your test set: {accuracy:.2f}%")


Evaluated Images: 480
Pre-trained Model Accuracy on your test set: 88.75%


## 2. EfficientNetB0


In [23]:
from tensorflow.keras.applications import EfficientNetB0

# 1. Load the EfficientNetB0 base model
# Note: EfficientNet models have built-in rescaling layers, but since we scale 1./255
# in train_generator, we can safely pass it as is.
base_efficientnet = EfficientNetB0(input_shape=(128, 128, 3),
                                    include_top=False,
                                    weights='imagenet')


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [24]:
# Freeze the base model
base_efficientnet.trainable = False

In [25]:
# 2. Build the Model Architecture
efficientnet_model = models.Sequential([
    base_efficientnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

In [26]:
# 3. Compile the model
efficientnet_model.compile(optimizer='adam',
                           loss='binary_crossentropy',
                           metrics=['accuracy'])

In [27]:
efficientnet_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,213,668 (16.07 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [28]:
# Train EfficientNetB0 with limited steps to train much faster
print("Training EfficientNetB0...")

efficientnet_callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
    ModelCheckpoint('best_efficientnet.keras', monitor='val_accuracy', save_best_only=True)
]


Training EfficientNetB0...


In [ ]:
efficientnet_history = efficientnet_model.fit(
    train_generator,
    steps_per_epoch=100,      # Only train on 100 batches (3200 images) per epoch
    epochs=10,                # Reduced from 15 to 10 to speed things up further
    validation_data=test_generator,
    validation_steps=20,      # Only validate on 20 batches (640 images) per epoch
    callbacks=efficientnet_callbacks
)

## 2.1 Load and Use EfficientNet Pretrained Model (As Is)


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

In [30]:
# 1. Load EfficientNetB0 fully pre-trained on ImageNet (as is)
pretrained_efficientnet_as_is = EfficientNetB0(weights='imagenet', include_top=True)
print("EfficientNetB0 loaded successfully with original classifier layers!")

def predict_image_efficientnet(img_path):
    # 2. Load and resize the image to 224x224 (required by default EfficientNetB0)
    img = image.load_img(img_path, target_size=(224, 224))

    # 3. Convert image to array and add batch dimension
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)

    # 4. Preprocess input according to EfficientNet requirements.txt
    x = preprocess_input(x)

    # 5. Run inference
    preds = pretrained_efficientnet_as_is.predict(x)

    # 6. Decode and print the top 3 predictions
    print('Predicted classes:', decode_predictions(preds, top=3)[0])


21834768/21834768 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
EfficientNetB0 loaded successfully with original classifier layers!


In [31]:
predict_image_efficientnet('/content/drive/MyDrive/CNN/test/cats/cat.10.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step
Predicted classes: [('n02123045', 'tabby', np.float32(0.24582452)), ('n02127052', 'lynx', np.float32(0.23134738)), ('n02124075', 'Egyptian_cat', np.float32(0.21383171))]


In [32]:
# 1. Load fully pre-trained EfficientNetB0 model on ImageNet
print("Loading pre-trained ImageNet EfficientNetB0 model...")
pretrained_efficientnet = EfficientNetB0(weights='imagenet', include_top=True)

# ImageNet indices for cats (278 to 287) and dogs (151 to 268)
cat_indices = set(range(278, 288))
dog_indices = set(range(151, 269))

correct_predictions = 0
total_predictions = 0

# Evaluate on 15 batches (480 images)
num_batches_to_eval = 20

print("Evaluating the pre-trained EfficientNetB0 model on your test dataset...")
for i in range(num_batches_to_eval):
    try:
        imgs, labels = next(test_generator_raw if 'test_generator_raw' in globals() else test_generator)
    except:
        break

    # EfficientNet expects 224x224 input. Re-scale back to 0-255 first, resize, then preprocess
    resized_imgs = tf.image.resize(imgs * 255.0, (224, 224))
    preprocessed_imgs = preprocess_input(resized_imgs.numpy())

    # Predict
    preds = pretrained_efficientnet.predict(preprocessed_imgs, verbose=0)

    for j in range(len(labels)):
        predicted_class_idx = np.argmax(preds[j])

        # Map ImageNet class to binary
        if predicted_class_idx in cat_indices:
            binary_pred = 0
        elif predicted_class_idx in dog_indices:
            binary_pred = 1
        else:
            cat_prob = sum(preds[j][idx] for idx in cat_indices if idx < len(preds[j]))
            dog_prob = sum(preds[j][idx] for idx in dog_indices if idx < len(preds[j]))
            binary_pred = 0 if cat_prob >= dog_prob else 1

        if binary_pred == int(labels[j]):
            correct_predictions += 1
        total_predictions += 1

eff_accuracy = (correct_predictions / total_predictions) * 100 if total_predictions > 0 else 0
print(f"\nEvaluated Images: {total_predictions}")
print(f"Pre-trained EfficientNetB0 Accuracy on your test set: {eff_accuracy:.2f}%")

Loading pre-trained ImageNet EfficientNetB0 model...
Evaluating the pre-trained EfficientNetB0 model on your test dataset...

Evaluated Images: 640
Pre-trained EfficientNetB0 Accuracy on your test set: 90.78%


### 3. Deploying Your Models in Streamlit: Saving vs. Direct Loading

When building your Streamlit application, you have two options depending on which model you choose to deploy:

#### Option A: Using the Raw Pre-trained Models (No Manual Saving Needed!)
If you choose to deploy the **raw, pre-trained ImageNet models** (which gave you **90.78%** accuracy with EfficientNetB0), you **do not need to save or download any model files**!

When your Streamlit app starts up on a machine or server, Keras automatically downloads the model weights from Google's servers. You can load it directly with a single line of code:
```python
import tensorflow as tf
model = tf.keras.applications.EfficientNetB0(weights='imagenet', include_top=True)
```

---

#### Option B: Using Your Fine-Tuned Custom Models (Saving Needed!)
If you prefer to deploy the models we fine-tuned on your custom dataset, they **must** be saved as files. Fortunately, our `ModelCheckpoint` callbacks **automatically saved them** during training!

You can find these files in your Colab file explorer on the left:
* `best_mobilenet.keras`
* `best_efficientnet.keras`

**What to do:**
1. Open the file browser in Google Colab (click the folder icon on the left panel).
2. Download `best_efficientnet.keras` to your local computer.
3. Put this file in the same folder as your Streamlit `app.py` script.
4. Load it inside your Streamlit app like this:
```python
import tensorflow as tf
model = tf.keras.models.load_model('best_efficientnet.keras')
```